# CNN MNIST Classifier for handwritten digits digits

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

In [2]:
torch.manual_seed(42)

transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainData = datasets.MNIST(root='./data', train=True, download=True, transform=transformer)
testData = datasets.MNIST(root='./data', train=False, download=True, transform=transformer)

train_loader = DataLoader(trainData, batch_size=64, shuffle=True)
test_loader = DataLoader(testData, batch_size=1000, shuffle=False)

In [ ]:
class CNN_MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolutional_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), #28+1*2-3+1 = 28x28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), #14x14
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1), #14x14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2), #7x7
            nn.Dropout2d(0.25) #prevents overfitting
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.convolutional_block(x)
        x = self.classifier(x)
        return x

In [20]:
model = CNN_MNIST()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Total Parameters: {sum(p.numel() for p in model.parameters())}")

Total Parameters: 421834


In [21]:
for epoch in range(5):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Average Loss Epoch {epoch+1}: {total_loss/len(train_loader):.4f}")

Average Loss Epoch 1: 0.2551
Average Loss Epoch 2: 0.1217
Average Loss Epoch 3: 0.0982
Average Loss Epoch 4: 0.0829
Average Loss Epoch 5: 0.0744


In [23]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        prediction = output.argmax(dim=1)
        correct += (prediction == y_batch).sum().item()
        total += y_batch.size(0)
print(f"\nTest accuracy: {100 * correct / total:.2f}%")


Test accuracy: 99.03%


In [24]:
PATH = "MNIST_CNNmodel_weights.pth"
torch.save(model.state_dict(), PATH)
print(f"Model state_dict saved successfully to {PATH}")

Model state_dict saved successfully to MNIST_CNNmodel_weights.pth
